In [0]:
# Silver feature table from Day 2
features_table = "workspace.ecommerce.user_features_silver"

# Bronze/clean events Delta from Day 1
events_path = "/Volumes/workspace/ecommerce/ecommerce_delta"

features_df = spark.read.table(features_table)
events_df = spark.read.format("delta").load(events_path)

print("Tables loaded")

In [0]:
from pyspark.sql import functions as F

purchase_label_df = (
    events_df
    .filter(F.col("event_type") == "purchase")
    .select("user_id")
    .distinct()
    .withColumn("label", F.lit(1))
)

print("Purchase labels created")

In [0]:
training_df = (
    features_df
    .join(purchase_label_df, on="user_id", how="left")
    .fillna({"label": 0})
)

training_df.show(5)

In [0]:
selected_cols = [
    "user_id",
    "total_events",
    "total_purchases",
    "total_spent",
    "avg_price",
    "unique_products",
    "label"
]

training_df = training_df.select(selected_cols)

In [0]:
training_df.groupBy("label").count().show()

In [0]:
total = training_df.count()

training_df.groupBy("label") \
    .agg((F.count("*")/total).alias("ratio")) \
    .show()

In [0]:
train_df = training_df.sampleBy(
    "label",
    fractions={0: 0.8, 1: 0.8},
    seed=42
)

test_df = training_df.subtract(train_df)

print("Train/Test split completed")

In [0]:
print("Train distribution")
train_df.groupBy("label").count().show()

print("Test distribution")
test_df.groupBy("label").count().show()

In [0]:
train_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.train_dataset")

test_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.test_dataset")

print("Train/Test datasets saved")

In [0]:
train_df.groupBy("label").count().display()